# Summed 1.2 GHz + 60 MHz offset-PDH drive

DAC229/2 → broadband RF amplifier → phase EOM. DAC228/2 → PDH demodulator LO.
The two digital outputs have independently adjustable amplitudes; the reference also has adjustable phase.
Use the matching **PDH** bit/HWH. Perform your normal RFSoC clock initialization and DAC-tile startup first.
Loading a bitstream interrupts all existing functions; attach without download only when this matching image is already running.
The old RF-chain FRA and the old DAC228/2 frequency-waveform output are replaced in this build.
The two feedback PLLs, intensity controller and closed-loop FRA remain available.


In [ ]:
import importlib
from pathlib import Path
from pynq import Overlay
import pdh_tones as pdh

importlib.invalidate_caches()
pdh = importlib.reload(pdh)
# If ol already exists, keep using it. Otherwise attach to the matching file:
# ol = Overlay('/path/to/PDH_1200_60/TransportPhaseLockFF.bit', download=False)
print('Helper:', pdh.__file__)
pdh.check(ol)
print(pdh.status(ol))


## Configure once, with amplitudes zero
This configures **only** DAC229/2 and DAC228/2. It sets the first mixer to **630 MHz**, not 1.2 GHz: the ±570 MHz digital tones become 60 MHz and 1.2 GHz after mixing.
The reference DAC uses its real waveform with the mixer bypassed. Sample rates and interpolation are unchanged.
`configure()` resets the source RFDC NCO; repeat the demodulation-phase calibration after this or a converter restart.


In [ ]:
print(pdh.configure(ol))  # All requested amplitudes default to zero.


## Check frequencies and levels before optimizing the optical lock
The following are low digital test levels, not calibrated voltages. Check available RF gain and the measured spectrum at the EOM input.
The sum of the carrier and 60 MHz digital amplitudes must be ≤1; reference amplitude has its own limit.
No feedback-controller settings are changed by these calls.


In [ ]:
carrier_amplitude = 0.10
sideband_amplitude = None  # Default: 0.587654 * carrier_amplitude; 0 disables it.
reference_amplitude = 0.10
reference_phase_deg = 0.0

print(pdh.set_levels(
    ol, carrier_amplitude=carrier_amplitude, sideband_amplitude=sideband_amplitude,
    reference_amplitude=reference_amplitude, reference_phase_deg=reference_phase_deg,
))


## Optimize the offset and PDH modulation depths
For the ideal maximum discriminator slope, target optical phase modulation indices βoffset≈1.84 and βPDH≈1.08 rad.
With β=πVpeak/Vπ, these require **0.586 Vπ peak at 1.2 GHz** and **0.344 Vπ peak at 60 MHz**.
If RF voltage transfer and Vπ were equal, the voltage ratio would be 0.588 (−4.62 dB in RF power).
Digital weights 0.55 and 0.3232 have that ratio, but do not establish the absolute modulation depths: calibrate at the EOM.

1. Set `sideband_amplitude=0`; maximize the selected +1.2 GHz optical sideband by adjusting carrier amplitude / RF gain.
2. Raise `sideband_amplitude` and maximize measured PDH slope while optimizing demodulation phase.
3. Check that amplifier compression, unwanted cavity resonances and residual amplitude modulation do not worsen lock noise.

The reference-output amplitude only sets demodulator LO drive; it is independent of optical modulation depth.
See `docs/pdh_tones.md` for the optical-spectrum derivation and physical assumptions.


## Adjust demodulation phase without restarting oscillators
Positive phase advances the reference cosine. The actual analog phase includes tile, filter and cable offsets.
Use this call for live tuning; do not repeatedly call `configure()` or change the RFDC mixer frequency.
Large changes are coefficient steps, not ramps, and can disturb a lock.


In [ ]:
reference_phase_deg = 35.0
print(pdh.set_reference_phase(ol, reference_phase_deg))


## Inspect state or mute
Any sticky clipping/streaming fault mutes both outputs. Inspect and fix its cause, then use `configure()` to rearm.
The nominal frequencies share the existing clock reference, but absolute analog phase across the DAC tiles is not guaranteed to repeat after a restart.


In [ ]:
print(pdh.status(ol))


In [ ]:
# Run when you want both PDH outputs off; other channels are unaffected.
pdh.stop(ol)
